# Neural Network

In [ ]:
import warnings
warnings.filterwarnings('ignore')

In [ ]:
import numpy as np
import pandas as pd
import csv
import os
import psycopg2
import boto3
from config import *
from pprint import pprint
from collections import Counter
import tensorflow as tf
from sklearn.preprocessing import StandardScaler,OneHotEncoder
from sklearn.model_selection import train_test_split

In [ ]:
def download_file_using_client():
    s3 = boto3.client('s3',
                     aws_access_key_id=ACCESS_ID,
                     aws_secret_access_key=ACCESS_KEY)
    s3.download_file('cagurr-capstone', 'final_data.csv', 'Resources/final_data.csv')
    
download_file_using_client()

In [ ]:
cleaned_crime = pd.read_csv('Resources/final_data.csv')
cleaned_crime.head()

In [ ]:
X_raw = cleaned_crime.drop(columns=['chipotle'],axis=1)

In [ ]:
X_raw.corr()

In [ ]:
cleaned_crime = cleaned_crime.drop(columns=['index', 'primary_type', 'latitude', 'longitude', 'ward'])
cleaned_crime = cleaned_crime.dropna()
cleaned_crime['population_density'] = cleaned_crime['population_density'].str.replace(',', '')
cleaned_crime['population_density'] = cleaned_crime['population_density'].astype('float')
cleaned_crime['population'] = cleaned_crime['population'].str.replace(',', '')
cleaned_crime['population'] = cleaned_crime['population'].astype('int')
cleaned_crime

In [ ]:
# Split our preprocessed data into our features and target arrays
X = cleaned_crime.drop(columns=['chipotle','population','rankings'],axis=1).values
y = cleaned_crime['chipotle'].values

# Split the preprocessed data into a training and testing dataset
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = .3, random_state=78, shuffle=True)

In [ ]:
# Create a StandardScaler instances
scaler = StandardScaler()

# Fit the StandardScaler
X_scaler = scaler.fit(X_train)

# Scale the data
X_train_scaled = X_scaler.transform(X_train)
X_test_scaled = X_scaler.transform(X_test)

In [ ]:
# Define the model - deep neural net, i.e., the number of input features and hidden nodes for each layer.
number_input_features = len(X_train[0])
hidden_nodes_layer1 = 15
hidden_nodes_layer2 = 10

nn = tf.keras.models.Sequential()

# First hidden layer
nn.add(
    tf.keras.layers.Dense(units=hidden_nodes_layer1, input_dim=number_input_features, activation="relu")
)


# Second hidden layer
nn.add(
    tf.keras.layers.Dense(units=hidden_nodes_layer2, activation="relu")
)

# Third hidden layer
nn.add(
    tf.keras.layers.Dropout(.2)
)

# Output layer
nn.add(tf.keras.layers.Dense(units=1, activation="sigmoid"))

# Check the structure of the model
nn.summary()

In [ ]:
# Import checkpoint dependencies
import os
from tensorflow.keras.callbacks import ModelCheckpoint

# Define the checkpoint path and filenames
os.makedirs("checkpoints/",exist_ok=True)
checkpoint_path = "checkpoints/weights_opt_1.{epoch:02d}.hdf5"

In [ ]:
# Compile the model
nn.compile(loss="binary_crossentropy", optimizer="adam", metrics=["accuracy"])

# Create a callback that saves the model's weights every epoch
cp_callback = ModelCheckpoint(
    filepath=checkpoint_path,
    verbose=1,
    save_weights_only=True,
    save_freq='epoch',
    period=5
)

# Train the model
fit_model = nn.fit(X_train_scaled, y_train, epochs=50, batch_size = 32, callbacks=[cp_callback])

In [ ]:
# Evaluate the model using the test data
model_loss, model_accuracy = nn.evaluate(X_test_scaled,y_test,verbose=2)
print(f"Loss: {model_loss}, Accuracy: {model_accuracy}")